In [2]:
"""
================================================================================
XGBoost vs TabNet for Financial Time Series Classification
================================================================================

This script compares two powerful models for tabular financial 

1. XGBoost (Extreme Gradient Boosting)
2. TabNet (Deep Learning for Tabular Data)

Task: Predict whether SPY will fall more than 5% over the next 20 trading days
      based on current market features.

================================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import torch
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, 
    roc_auc_score, 
    precision_score, 
    recall_score, 
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)
from xgboost import XGBClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt

# ==============================================================================
# Configuration
# ==============================================================================

STOCK_TICKERS = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "META",
    "NVDA", "JPM", "BAC", "GS", "TSLA"
]

MARKET_TICKER = "SPY"
SECTOR_TICKER = "XLK"
MACRO_TICKER = "TLT"

DRAWDOWN_THRESHOLD = -0.05
FORECAST_HORIZON = 20

RANDOM_STATE = 42
TEST_SIZE = 0.25

# Set matplotlib backend for PNG saving
plt.switch_backend('Agg')

# ==============================================================================
# 1. Data Loading
# ==============================================================================

def download_market_data():
    """Download historical price and volume data for all tickers."""
    all_tickers = STOCK_TICKERS + [MARKET_TICKER, SECTOR_TICKER, MACRO_TICKER]
    
    print("Downloading market data...")
    raw = yf.download(
        all_tickers,
        start="2018-01-01",
        auto_adjust=True,
        progress=True
    )
    
    close = raw["Close"].copy()
    volume = raw["Volume"].copy()
    
    print(f"Data shape: {close.shape}")
    print(close.tail())
    
    return close, volume

def separate_benchmarks(close, volume):
    """Separate individual stock prices from benchmark/ETF prices."""
    stock_prices = close[STOCK_TICKERS].dropna(how="all")
    stock_volume = volume[STOCK_TICKERS].dropna(how="all")
    
    spy = close[MARKET_TICKER].dropna()
    sector_px = close[SECTOR_TICKER].dropna()
    macro_px = close[MACRO_TICKER].dropna()
    
    return stock_prices, stock_volume, spy, sector_px, macro_px

# ==============================================================================
# 2. Feature Engineering
# ==============================================================================

def compute_returns(stock_prices, spy, sector_px, macro_px):
    """Compute daily returns for all price series."""
    stock_returns = stock_prices.pct_change().dropna(how="all")
    spy_ret = spy.pct_change().dropna()
    sector_ret = sector_px.pct_change().dropna()
    macro_ret = macro_px.pct_change().dropna()
    
    return stock_returns, spy_ret, sector_ret, macro_ret

def build_feature_table(stock_prices, stock_volume, stock_returns, 
                        spy, sector_px, macro_px):
    """Build the core feature table with cross-sectional and macro features."""
    features = pd.DataFrame(index=stock_returns.index)
    
    features["mean_return"] = stock_returns.mean(axis=1)
    features["volatility"] = stock_returns.std(axis=1)
    features["momentum_20"] = stock_prices.pct_change(20).mean(axis=1)
    features["momentum_60"] = stock_prices.pct_change(60).mean(axis=1)
    
    drawdown = stock_prices / stock_prices.cummax() - 1
    features["drawdown"] = drawdown.mean(axis=1)
    
    vol_short = stock_volume.rolling(20).mean()
    vol_long = stock_volume.rolling(60).mean()
    volume_trend = (vol_short / vol_long).replace([np.inf, -np.inf], np.nan)
    features["volume_trend"] = volume_trend.mean(axis=1)
    
    sector_ret = sector_px.pct_change().dropna()
    window_beta = 60
    sector_beta_list = []
    
    for tkr in STOCK_TICKERS:
        aligned = pd.concat(
            [stock_returns[tkr], sector_ret],
            axis=1,
            join="inner"
        ).dropna()
        
        aligned.columns = ["stock_ret", "sector_ret"]
        
        cov = aligned["stock_ret"].rolling(window_beta).cov(aligned["sector_ret"])
        var = aligned["sector_ret"].rolling(window_beta).var()
        
        beta = cov / var
        sector_beta_list.append(beta.rename(tkr))
    
    sector_beta_df = pd.concat(sector_beta_list, axis=1)
    features["sector_beta"] = sector_beta_df.mean(axis=1)
    
    print("Downloading additional macro data (GLD, DBC)...")
    gld = yf.download("GLD", start="2018-01-01", auto_adjust=True, progress=False)["Close"]
    dbc = yf.download("DBC", start="2018-01-01", auto_adjust=True, progress=False)["Close"]
    
    if isinstance(gld, pd.DataFrame):
        gld = gld.iloc[:, 0]
    if isinstance(dbc, pd.DataFrame):
        dbc = dbc.iloc[:, 0]
    
    gld = gld.reindex(features.index)
    dbc = dbc.reindex(features.index)
    macro_px_aligned = macro_px.reindex(features.index)
    
    macro_combo = pd.DataFrame({
        "TLT_20d": macro_px_aligned.pct_change(20),
        "GLD_20d": gld.pct_change(20),
        "DBC_20d": dbc.pct_change(20)
    })
    
    features["macro_factor"] = macro_combo.mean(axis=1)
    
    spy_ret_20d = spy.pct_change(20)
    features["future_20d_return"] = spy_ret_20d.shift(-20)
    features["future_20d_drawdown_flag"] = (
        features["future_20d_return"] < DRAWDOWN_THRESHOLD
    ).astype(float)
    
    features = features.dropna(subset=["future_20d_return"]).copy()
    features["future_20d_drawdown_flag"] = features["future_20d_drawdown_flag"].astype(int)
    
    print(f"Feature table shape: {features.shape}")
    print(f"Class distribution: {features['future_20d_drawdown_flag'].value_counts().to_dict()}")
    
    return features

# ==============================================================================
# 3. XGBoost Model
# ==============================================================================

def train_xgboost(X_train, X_test, y_train, y_test):
    """Train XGBoost classifier and return model with predictions."""
    print("\n" + "="*80)
    print("Training XGBoost Classifier")
    print("="*80)
    
    n_neg = np.sum(y_train == 0)
    n_pos = np.sum(y_train == 1)
    scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0
    
    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight
    )
    
    xgb.fit(X_train, y_train)
    
    pred_class = xgb.predict(X_test)
    pred_prob = xgb.predict_proba(X_test)[:, 1]
    
    print("\nXGBoost Results:")
    print("Accuracy:", accuracy_score(y_test, pred_class))
    print("ROC AUC:", roc_auc_score(y_test, pred_prob))
    print(classification_report(y_test, pred_class, zero_division=0))
    
    return xgb, pred_class, pred_prob

def get_xgboost_feature_importance(model, feature_cols):
    """Extract feature importance from XGBoost model."""
    xgb_imp = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
    
    return xgb_imp

# ==============================================================================
# 4. TabNet Model
# ==============================================================================

def prepare_tabnet_data(X_train, X_test, y_train, y_test):
    """Prepare data for TabNet (handle NaNs, convert to numpy)."""
    imputer = SimpleImputer(strategy="median")
    
    X_train_np = imputer.fit_transform(X_train)
    X_test_np = imputer.transform(X_test)
    
    y_train_np = y_train.to_numpy()
    y_test_np = y_test.to_numpy()
    
    print("\nNaNs in X_train after impute:", np.isnan(X_train_np).sum())
    print("NaNs in X_test after impute:", np.isnan(X_test_np).sum())
    
    return X_train_np, X_test_np, y_train_np, y_test_np

def create_sample_weights(y_train):
    """Create sample weights for TabNet to handle class imbalance."""
    n_neg = np.sum(y_train == 0)
    n_pos = np.sum(y_train == 1)
    
    weights = np.ones(len(y_train))
    
    if n_pos > 0:
        class_weight = n_neg / n_pos
        weights[y_train == 1] = class_weight
    
    print(f"Class weights - Negative: 1.0, Positive: {n_neg/n_pos if n_pos > 0 else 1.0:.2f}")
    print(f"Weight array shape: {weights.shape}")
    
    return weights

def train_tabnet(X_train_np, X_test_np, y_train_np, y_test_np):
    """Train TabNet classifier and return model with predictions."""
    print("\n" + "="*80)
    print("Training TabNet Classifier")
    print("="*80)
    
    sample_weights = create_sample_weights(y_train_np)
    
    tabnet = TabNetClassifier(
        n_d=16,
        n_a=16,
        n_steps=5,
        gamma=1.5,
        lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        mask_type="entmax",
        seed=RANDOM_STATE
    )
    
    tabnet.fit(
        X_train_np,
        y_train_np,
        eval_set=[(X_test_np, y_test_np)],
        eval_name=["test"],
        eval_metric=["auc"],
        max_epochs=1000,
        patience=15,
        batch_size=256,
        virtual_batch_size=128,
        weights=sample_weights
    )
    
    pred_class = tabnet.predict(X_test_np)
    pred_prob = tabnet.predict_proba(X_test_np)[:, 1]
    
    print("\nTabNet Results:")
    print("Accuracy:", accuracy_score(y_test_np, pred_class))
    print("ROC AUC:", roc_auc_score(y_test_np, pred_prob))
    print(confusion_matrix(y_test_np, pred_class))
    print(classification_report(y_test_np, pred_class, zero_division=0))
    
    return tabnet, pred_class, pred_prob

def get_tabnet_feature_importance(model, feature_cols):
    """Extract feature importance from TabNet model."""
    tabnet_imp = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
    
    return tabnet_imp

# ==============================================================================
# 5. Evaluation & Visualization
# ==============================================================================

def compare_models(xgb_metrics, tabnet_metrics):
    """Create comparison table of model metrics."""
    comparison = pd.DataFrame({
        "Model": ["XGBoost", "TabNet"],
        "Accuracy": [xgb_metrics["accuracy"], tabnet_metrics["accuracy"]],
        "ROC_AUC": [xgb_metrics["auc"], tabnet_metrics["auc"]],
        "Precision": [xgb_metrics["precision"], tabnet_metrics["precision"]],
        "Recall": [xgb_metrics["recall"], tabnet_metrics["recall"]],
        "F1": [xgb_metrics["f1"], tabnet_metrics["f1"]]
    })
    
    comparison = comparison.round(4)
    print("\n" + "="*80)
    print("Model Comparison")
    print("="*80)
    print(comparison)
    
    return comparison

def export_figure_html(fig, base_filename):
    """Export Plotly figure as HTML."""
    html_filename = f"{base_filename}.html"
    fig.write_html(html_filename)
    print(f"✓ Saved {html_filename}")
    return html_filename

def save_roc_curve_png(y_test, pred_prob, model_name, filename):
    """Save ROC curve as PNG using matplotlib (RELIABLE)."""
    fpr, tpr, _ = roc_curve(y_test, pred_prob)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(10, 7))
    plt.plot(fpr, tpr, color='darkorange', lw=2, 
             label=f'{model_name} ROC Curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', 
             label='Random Guess')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} ROC Curve')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved {filename}")
    return filename

def save_feature_importance_png(importance_df, model_name, filename):
    """Save feature importance as PNG using matplotlib (RELIABLE)."""
    plt.figure(figsize=(10, 7))
    plt.barh(range(len(importance_df)), importance_df['importance'].values)
    plt.yticks(range(len(importance_df)), importance_df['feature'].values)
    plt.xlabel('Importance')
    plt.title(f'{model_name} Feature Importance')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved {filename}")
    return filename

def save_model_comparison_png(comparison_df, filename):
    """Save model comparison as PNG using matplotlib (RELIABLE)."""
    metrics = ['Accuracy', 'ROC_AUC', 'Precision', 'Recall', 'F1']
    x = np.arange(len(metrics))
    width = 0.35
    
    xgb_vals = [comparison_df[comparison_df['Model']=='XGBoost'][m].values[0] for m in metrics]
    tab_vals = [comparison_df[comparison_df['Model']=='TabNet'][m].values[0] for m in metrics]
    
    fig, ax = plt.subplots(figsize=(12, 7))
    rects1 = ax.bar(x - width/2, xgb_vals, width, label='XGBoost', color='steelblue')
    rects2 = ax.bar(x + width/2, tab_vals, width, label='TabNet', color='coral')
    
    ax.set_ylabel('Score')
    ax.set_title('XGBoost vs TabNet Classifier Performance')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for rect in rects1 + rects2:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved {filename}")
    return filename

def save_feature_comparison_png(xgb_imp, tabnet_imp, filename):
    """Save feature importance comparison as PNG using matplotlib (RELIABLE)."""
    compare_imp = xgb_imp.merge(
        tabnet_imp,
        on="feature",
        how="outer",
        suffixes=("_xgb", "_tabnet")
    ).fillna(0)
    
    features = compare_imp['feature'].values
    x = np.arange(len(features))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(12, 9))
    rects1 = ax.barh(x - width/2, compare_imp['importance_xgb'].values, width, 
                     label='XGBoost', color='steelblue')
    rects2 = ax.barh(x + width/2, compare_imp['importance_tabnet'].values, width, 
                     label='TabNet', color='coral')
    
    ax.set_xlabel('Importance')
    ax.set_title('XGBoost vs TabNet Feature Importance')
    ax.set_yticks(x)
    ax.set_yticklabels(features)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved {filename}")
    return filename

def plot_roc_curve(y_test, pred_prob, model_name):
    """Plot ROC curve - exports both HTML (plotly) and PNG (matplotlib)."""
    fpr, tpr, _ = roc_curve(y_test, pred_prob)
    roc_auc = auc(fpr, tpr)
    
    # HTML export (plotly)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"{model_name} ROC Curve (AUC = {roc_auc:.3f})"
    ))
    fig.add_trace(go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Random Guess",
        line=dict(dash="dash")
    ))
    
    fig.update_layout(
        title=f"{model_name} ROC Curve",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate",
        width=1100,
        height=600
    )
    
    base_filename = f"{model_name.lower()}_roc_curve"
    html_file = export_figure_html(fig, base_filename)
    
    # PNG export (matplotlib - RELIABLE)
    png_file = save_roc_curve_png(y_test, pred_prob, model_name, f"{base_filename}.png")
    
    return fig, [html_file, png_file]

def plot_feature_importance(importance_df, model_name):
    """Plot feature importance - exports both HTML (plotly) and PNG (matplotlib)."""
    fig = px.bar(
        importance_df,
        x="importance",
        y="feature",
        orientation="h",
        title=f"{model_name} Feature Importance"
    )
    
    fig.update_layout(
        width=1100,
        height=600,
        yaxis={"categoryorder": "total ascending"}
    )
    
    base_filename = f"{model_name.lower()}_feature_importance"
    html_file = export_figure_html(fig, base_filename)
    
    # PNG export (matplotlib - RELIABLE)
    png_file = save_feature_importance_png(importance_df, model_name, f"{base_filename}.png")
    
    return fig, [html_file, png_file]

def plot_model_comparison(comparison_df):
    """Plot model comparison - exports both HTML (plotly) and PNG (matplotlib)."""
    comparison_long = comparison_df.melt(
        id_vars="Model",
        value_vars=["Accuracy", "ROC_AUC", "Precision", "Recall", "F1"],
        var_name="Metric",
        value_name="Score"
    )
    
    fig = px.bar(
        comparison_long,
        x="Metric",
        y="Score",
        color="Model",
        barmode="group",
        text="Score",
        title="XGBoost vs TabNet Classifier Performance"
    )
    
    fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
    
    fig.update_layout(
        width=1100,
        height=600,
        yaxis_title="Score",
        xaxis_title="Metric",
        yaxis=dict(range=[0, 1]),
        legend_title="Model"
    )
    
    html_file = export_figure_html(fig, "model_comparison")
    
    # PNG export (matplotlib - RELIABLE)
    png_file = save_model_comparison_png(comparison_df, "model_comparison.png")
    
    return fig, [html_file, png_file]

def plot_feature_importance_comparison(xgb_imp, tabnet_imp):
    """Plot feature comparison - exports both HTML (plotly) and PNG (matplotlib)."""
    compare_imp = xgb_imp.merge(
        tabnet_imp,
        on="feature",
        how="outer",
        suffixes=("_xgb", "_tabnet")
    ).fillna(0)
    
    compare_long = compare_imp.melt(
        id_vars="feature",
        value_vars=["importance_xgb", "importance_tabnet"],
        var_name="model",
        value_name="importance"
    )
    
    fig = px.bar(
        compare_long,
        x="importance",
        y="feature",
        color="model",
        barmode="group",
        orientation="h",
        title="XGBoost vs TabNet Feature Importance"
    )
    
    fig.update_layout(
        width=1100,
        height=800,
        yaxis={"categoryorder": "total ascending"}
    )
    
    html_file = export_figure_html(fig, "feature_importance_comparison")
    
    # PNG export (matplotlib - RELIABLE)
    png_file = save_feature_comparison_png(xgb_imp, tabnet_imp, "feature_importance_comparison.png")
    
    return fig, [html_file, png_file]

# ==============================================================================
# 6. Main Execution
# ==============================================================================

def main():
    """Main execution function."""
    print("\n" + "="*80)
    print("XGBoost vs TabNet for Financial Time Series Classification")
    print("="*80)
    
    all_exported_files = []
    
    # -------------------------------------------------------------------------
    # Step 1: Load Data
    # -------------------------------------------------------------------------
    print("\n--- Step 1: Loading Data ---")
    close, volume = download_market_data()
    
    # -------------------------------------------------------------------------
    # Step 2: Separate Benchmarks
    # -------------------------------------------------------------------------
    print("\n--- Step 2: Separating Benchmarks ---")
    stock_prices, stock_volume, spy, sector_px, macro_px = separate_benchmarks(
        close, volume
    )
    
    # -------------------------------------------------------------------------
    # Step 3: Compute Returns
    # -------------------------------------------------------------------------
    print("\n--- Step 3: Computing Returns ---")
    stock_returns, spy_ret, sector_ret, macro_ret = compute_returns(
        stock_prices, spy, sector_px, macro_px
    )
    
    # -------------------------------------------------------------------------
    # Step 4: Build Feature Table
    # -------------------------------------------------------------------------
    print("\n--- Step 4: Building Feature Table ---")
    features = build_feature_table(
        stock_prices, stock_volume, stock_returns, 
        spy, sector_px, macro_px
    )
    
    # -------------------------------------------------------------------------
    # Step 5: Prepare Train/Test Split
    # -------------------------------------------------------------------------
    print("\n--- Step 5: Train/Test Split ---")
    feature_cols = [
        "mean_return",
        "volatility",
        "momentum_20",
        "momentum_60",
        "drawdown",
        "volume_trend",
        "sector_beta",
        "macro_factor"
    ]
    
    X = features[feature_cols].copy()
    y = features["future_20d_drawdown_flag"].copy()
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    
    print(f"Training set size: {len(X_train)}")
    print(f"Test set size: {len(X_test)}")
    print(f"Class distribution (train): {y_train.value_counts().to_dict()}")
    print(f"Class distribution (test): {y_test.value_counts().to_dict()}")
    
    # -------------------------------------------------------------------------
    # Step 6: Train XGBoost
    # -------------------------------------------------------------------------
    print("\n--- Step 6: Training XGBoost ---")
    xgb_model, xgb_pred_class, xgb_pred_prob = train_xgboost(
        X_train, X_test, y_train, y_test
    )
    
    xgb_metrics = {
        "accuracy": accuracy_score(y_test, xgb_pred_class),
        "auc": roc_auc_score(y_test, xgb_pred_prob),
        "precision": precision_score(y_test, xgb_pred_class, zero_division=0),
        "recall": recall_score(y_test, xgb_pred_class, zero_division=0),
        "f1": f1_score(y_test, xgb_pred_class, zero_division=0)
    }
    
    xgb_imp = get_xgboost_feature_importance(xgb_model, feature_cols)
    
    # -------------------------------------------------------------------------
    # Step 7: Prepare TabNet Data
    # -------------------------------------------------------------------------
    print("\n--- Step 7: Preparing TabNet Data ---")
    X_train_np, X_test_np, y_train_np, y_test_np = prepare_tabnet_data(
        X_train, X_test, y_train, y_test
    )
    
    # -------------------------------------------------------------------------
    # Step 8: Train TabNet
    # -------------------------------------------------------------------------
    print("\n--- Step 8: Training TabNet ---")
    tabnet_model, tab_pred_class, tab_pred_prob = train_tabnet(
        X_train_np, X_test_np, y_train_np, y_test_np
    )
    
    tabnet_metrics = {
        "accuracy": accuracy_score(y_test_np, tab_pred_class),
        "auc": roc_auc_score(y_test_np, tab_pred_prob),
        "precision": precision_score(y_test_np, tab_pred_class, zero_division=0),
        "recall": recall_score(y_test_np, tab_pred_class, zero_division=0),
        "f1": f1_score(y_test_np, tab_pred_class, zero_division=0)
    }
    
    tabnet_imp = get_tabnet_feature_importance(tabnet_model, feature_cols)
    
    # -------------------------------------------------------------------------
    # Step 9: Compare Models
    # -------------------------------------------------------------------------
    print("\n--- Step 9: Comparing Models ---")
    comparison_df = compare_models(xgb_metrics, tabnet_metrics)
    
    # -------------------------------------------------------------------------
    # Step 10: Generate Visualizations
    # -------------------------------------------------------------------------
    print("\n--- Step 10: Generating Visualizations ---")
    
    _, files = plot_roc_curve(y_test, xgb_pred_prob, "XGBoost")
    all_exported_files.extend(files)
    
    _, files = plot_roc_curve(y_test_np, tab_pred_prob, "TabNet")
    all_exported_files.extend(files)
    
    _, files = plot_feature_importance(xgb_imp, "XGBoost")
    all_exported_files.extend(files)
    
    _, files = plot_feature_importance(tabnet_imp, "TabNet")
    all_exported_files.extend(files)
    
    _, files = plot_model_comparison(comparison_df)
    all_exported_files.extend(files)
    
    _, files = plot_feature_importance_comparison(xgb_imp, tabnet_imp)
    all_exported_files.extend(files)
    
    # -------------------------------------------------------------------------
    # Step 11: Summary
    # -------------------------------------------------------------------------
    print("\n" + "="*80)
    print("Analysis Complete!")
    print("="*80)
    
    html_files = [f for f in all_exported_files if f.endswith('.html')]
    png_files = [f for f in all_exported_files if f.endswith('.png')]
    
    print(f"\n✓ Exported {len(all_exported_files)} files:")
    print(f"  - {len(html_files)} HTML files (interactive)")
    print(f"  - {len(png_files)} PNG files (static)")
    
    print("\nHTML Files:")
    for f in html_files:
        print(f"  ✓ {f}")
    
    print("\nPNG Files:")
    for f in png_files:
        print(f"  ✓ {f}")
    
    print("="*80)

if __name__ == "__main__":
    main()

[                       0%                       ]


XGBoost vs TabNet for Financial Time Series Classification

--- Step 1: Loading Data ---


[*********************100%***********************]  13 of 13 completed


Data shape: (2056, 13)
Ticker            AAPL        AMZN        BAC       GOOGL          GS  \
Date                                                                    
2026-03-03  263.750000  208.729996  49.689102  303.366425  862.580017   
2026-03-04  262.519989  216.820007  50.017242  302.916779  867.250000   
2026-03-05  260.290009  218.940002  49.530003  300.668335  835.460022   
2026-03-06  257.459991  213.210007  48.639999  298.309998  821.419983   
2026-03-09  259.880005  213.490005  47.900002  306.359985  832.030029   

Ticker             JPM        META        MSFT        NVDA         SPY  \
Date                                                                     
2026-03-03  300.260010  655.080017  403.929993  180.050003  680.330017   
2026-03-04  299.390015  667.729980  405.200012  183.039993  685.130005   
2026-03-05  293.549988  660.570007  410.679993  183.339996  681.309998   
2026-03-06  289.480011  644.859985  408.959991  177.820007  672.380005   
2026-03-09  289.92001